# 11 — Model Baseline Comparison

Compare how different LLMs reproduce Day 0 survey responses from persona alone.

**Goal:** Find which model best recovers the real YouGov survey distribution,
measuring exact accuracy, ordinal accuracy, signed error (bias), MAE,
direction accuracy, and rank correlation.

**Design:** For each model, we administer the Day 0 survey to the same N agents
on one policy and compare LLM responses to the real survey ground truth.

In [1]:
# ── Imports ──
import sys, os, logging, time, random
import pandas as pd
import numpy as np
from statistics import mean, stdev
from collections import Counter
from scipy.stats import spearmanr

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from cag.io.survey import load
from cag.io.llm import load_api_key
from cag.abm.agent import SurveyedCitizen
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import (
    ClimatePolicyID, SURVEY_COLUMN_MAP, RESPONSE_SCALE,
    ordinal_score,
)

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

Imports OK


## Configuration

Edit the models list below. Each entry is `(display_name, model_id, provider)`.

In [2]:
# ── Experiment config ──
# (display_name, model_id, provider)
MODELS = [
    # Google Gemini
    ("gemini-2.5-flash",   "gemini-2.5-flash",         "genai"),
    ("gemini-3-flash",     "gemini-3-flash-preview",    "genai"),
    ("gemini-3.1-pro",     "gemini-3.1-pro-preview",    "genai"),
    # OpenAI GPT  (nano < mini < 5.4 flagship — no "pro" chat model exists)
    ("gpt-5.4-nano",      "gpt-5.4-nano",              "openai"),
    ("gpt-5.4-mini",      "gpt-5.4-mini",              "openai"),
    ("gpt-5.4",           "gpt-5.4",                    "openai"),
    # Anthropic Claude
    ("claude-haiku-4-5",   "claude-haiku-4-5",          "anthropic"),
    ("claude-sonnet-4-6",  "claude-sonnet-4-6",         "anthropic"),
    ("claude-opus-4-7",    "claude-opus-4-7",           "anthropic"),
]

POLICY_ID = ClimatePolicyID.BAN_PETROL_CARS   # Change to test other policies
N_AGENTS = 2         # Number of agents to test (same across all models)
TEMPERATURE = 0.5
THINKING = True        # Enable model thinking/reasoning for all providers
RANDOM_SEED = 44

# Load API keys
api_keys = {}
for _, _, provider in MODELS:
    if provider not in api_keys:
        api_keys[provider] = load_api_key(provider)

print(f"Policy: {POLICY_ID}")
print(f"Agents: {N_AGENTS}, Seed: {RANDOM_SEED}, Temperature: {TEMPERATURE}, Thinking: {THINKING}")
print(f"Models: {[name for name, _, _ in MODELS]}")
print(f"API keys loaded for: {list(api_keys.keys())}")

Policy: ClimatePolicyID(3)
Agents: 2, Seed: 44, Temperature: 0.5, Thinking: True
Models: ['gemini-2.5-flash', 'gemini-3-flash', 'gemini-3.1-pro', 'gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.4', 'claude-haiku-4-5', 'claude-sonnet-4-6', 'claude-opus-4-7']
API keys loaded for: ['genai', 'openai', 'anthropic']


## Build environment and sample agents

In [3]:
# ── Build the SurveyedNation and sample agents ──
random.seed(RANDOM_SEED)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
data = data.sample(n=N_AGENTS, random_state=RANDOM_SEED)

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

agents = list(sn.agents_active.values())
print(f"Sampled {len(agents)} agents")

# Ground truth for each agent
ground_truth = {}
for agent in agents:
    ground_truth[agent.id] = agent.get_real_survey_response(POLICY_ID)

gt_values = list(ground_truth.values())
print(f"Ground truth mean: {mean(gt_values):+.3f}, SD: {stdev(gt_values):.3f}")
print(f"Distribution: {dict(sorted(Counter(gt_values).items()))}")

Sampled 2 agents
Ground truth mean: -1.000, SD: 2.828
Distribution: {-3: 1, 1: 1}


## Run Day 0 survey for each model

This cell calls the LLM API for each model × agent combination.  
**Cost estimate:** ~N_AGENTS × len(MODELS) API calls, each a short prompt+response.

In [4]:
# ── Run Day 0 baseline for each model ──
all_results = []  # list of dicts

for model_name, model_id, provider in MODELS:
    print(f"\n{'='*60}")
    print(f"Running: {model_name} ({provider})")
    print(f"{'='*60}")
    
    api_key = api_keys[provider]
    t0 = time.time()
    
    for i, agent in enumerate(agents):
        # Clear any prior opinion history from previous model runs
        agent.opinion_history = {}
        
        try:
            letter, numeric = agent.administer_survey(
                POLICY_ID, day=0,
                model=model_id, provider=provider,
                api_key=api_key, temperature=TEMPERATURE,
                thinking=THINKING,
            )
            real = ground_truth[agent.id]
            
            all_results.append({
                "model": model_name,
                "agent_id": agent.id,
                "llm_letter": letter,
                "llm_numeric": numeric,
                "real_numeric": real,
                "error": numeric - real,
                "abs_error": abs(numeric - real),
                "exact_match": int(numeric == real),
                "ordinal_score": ordinal_score(numeric, real),
            })
            
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(agents)} agents done")
                
        except Exception as e:
            print(f"  ERROR agent {agent.id}: {e}")
            all_results.append({
                "model": model_name,
                "agent_id": agent.id,
                "llm_letter": None,
                "llm_numeric": None,
                "real_numeric": ground_truth[agent.id],
                "error": None,
                "abs_error": None,
                "exact_match": 0,
                "ordinal_score": None,
            })
    
    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")

df = pd.DataFrame(all_results)
print(f"\nTotal results: {len(df)} ({len(df[df['llm_numeric'].notna()])} successful)")


Running: gemini-2.5-flash (genai)
  Done in 26.9s

Running: gemini-3-flash (genai)
  Done in 12.5s

Running: gemini-3.1-pro (genai)
  Done in 12.0s

Running: gpt-5.4-nano (openai)


  Done in 4.7s

Running: gpt-5.4-mini (openai)


  Done in 3.4s

Running: gpt-5.4 (openai)


  Done in 7.5s

Running: claude-haiku-4-5 (anthropic)


  Done in 6.1s

Running: claude-sonnet-4-6 (anthropic)
  Done in 2.7s

Running: claude-opus-4-7 (anthropic)
  Done in 2.7s

Total results: 18 (18 successful)


## Compute accuracy metrics per model

| Metric | Description |
|---|---|
| **Exact accuracy** | % of agents where LLM == real survey response |
| **Ordinal score** | 1 − (distance / max_distance), averaged over agents. 1.0 = perfect, 0.0 = worst |
| **Mean signed error** | Average (LLM − real). Positive = pro-climate bias |
| **MAE** | Mean absolute error on the −3 to +3 scale |
| **Direction accuracy** | % where LLM gets the correct side: oppose (< 0), neutral (0), support (> 0) |
| **Spearman ρ** | Rank correlation between LLM and real responses. 1.0 = perfect ranking |

In [5]:
# ── Compute metrics per model ──
def direction(v):
    """Map numeric opinion to direction: -1=oppose, 0=neutral, +1=support."""
    if v < 0: return -1
    elif v > 0: return 1
    else: return 0

summary_rows = []
for model_name in df["model"].unique():
    mdf = df[(df["model"] == model_name) & df["llm_numeric"].notna()].copy()
    n = len(mdf)
    if n == 0:
        continue
    
    exact_acc = mdf["exact_match"].mean()
    ord_score = mdf["ordinal_score"].mean()
    mean_err = mdf["error"].mean()
    mae = mdf["abs_error"].mean()
    
    # Direction accuracy
    mdf["llm_dir"] = mdf["llm_numeric"].apply(direction)
    mdf["real_dir"] = mdf["real_numeric"].apply(direction)
    dir_acc = (mdf["llm_dir"] == mdf["real_dir"]).mean()
    
    # Spearman correlation
    if mdf["llm_numeric"].nunique() > 1 and mdf["real_numeric"].nunique() > 1:
        rho, p_val = spearmanr(mdf["llm_numeric"], mdf["real_numeric"])
    else:
        rho, p_val = float('nan'), float('nan')
    
    # Mean LLM response (to check distribution)
    mean_llm = mdf["llm_numeric"].mean()
    
    summary_rows.append({
        "Model": model_name,
        "n": n,
        "Mean LLM": mean_llm,
        "Exact %": exact_acc,
        "Ordinal": ord_score,
        "Mean Error": mean_err,
        "MAE": mae,
        "Direction %": dir_acc,
        "Spearman ρ": rho,
        "Spearman p": p_val,
    })

summary = pd.DataFrame(summary_rows)

# Ground truth reference row
gt_mean = mean(gt_values)
print(f"Ground truth mean: {gt_mean:+.3f}")
print()

# Display
display_cols = ["Model", "n", "Mean LLM", "Exact %", "Ordinal", "Mean Error", "MAE", "Direction %", "Spearman ρ"]
styled = summary[display_cols].style.format({
    "Mean LLM": "{:+.3f}",
    "Exact %": "{:.1%}",
    "Ordinal": "{:.3f}",
    "Mean Error": "{:+.3f}",
    "MAE": "{:.3f}",
    "Direction %": "{:.1%}",
    "Spearman ρ": "{:.3f}",
}).highlight_min(subset=["MAE", "Mean Error"], color="lightgreen"
).highlight_max(subset=["Exact %", "Ordinal", "Direction %", "Spearman ρ"], color="lightgreen")

styled

Ground truth mean: -1.000



,Model,n,Mean LLM,Exact %,Ordinal,Mean Error,MAE,Direction %,Spearman ρ
0,gemini-2.5-flash,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
1,gemini-3-flash,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
2,gemini-3.1-pro,2,+0.000,50.0%,0.833,+1.000,1.000,100.0%,1.000
3,gpt-5.4-nano,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
4,gpt-5.4-mini,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
5,gpt-5.4,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
6,claude-haiku-4-5,2,+1.000,0.0%,0.667,+2.000,2.000,100.0%,1.000
7,claude-sonnet-4-6,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000
8,claude-opus-4-7,2,+0.500,0.0%,0.750,+1.500,1.500,100.0%,1.000


## Error distribution by model

In [ ]:
# ── Error distribution heatmap ──
import matplotlib.pyplot as plt

models_list = df["model"].unique()
error_range = range(-6, 7)

fig, ax = plt.subplots(figsize=(12, max(3, len(models_list) * 0.8 + 1)))

heatmap_data = []
for model_name in models_list:
    mdf = df[(df["model"] == model_name) & df["error"].notna()]
    err_counts = Counter(mdf["error"].astype(int))
    row = [err_counts.get(e, 0) for e in error_range]
    heatmap_data.append(row)

heatmap_array = np.array(heatmap_data)
im = ax.imshow(heatmap_array, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(list(error_range))))
ax.set_xticklabels([f"{e:+d}" for e in error_range])
ax.set_yticks(range(len(models_list)))
ax.set_yticklabels(models_list)
ax.set_xlabel("Error (LLM − Real)")
ax.set_title("Error Distribution by Model")

# Annotate cells
for i in range(len(models_list)):
    for j in range(len(list(error_range))):
        val = heatmap_array[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    color='white' if val > heatmap_array.max() * 0.5 else 'black', fontsize=9)

plt.colorbar(im, ax=ax, label="Count")
plt.tight_layout()
plt.show()

## Response distribution comparison

In [ ]:
# ── Response distribution: each model vs ground truth ──
scale_values = list(range(-3, 4))
n_models = len(models_list)
fig, axes = plt.subplots(1, n_models + 1, figsize=(4 * (n_models + 1), 4), sharey=True)

# Ground truth
gt_dist = Counter(gt_values)
gt_pcts = [gt_dist.get(v, 0) / len(gt_values) * 100 for v in scale_values]
axes[0].bar(scale_values, gt_pcts, color='steelblue', alpha=0.8)
axes[0].set_title(f'Ground Truth\n(mean={mean(gt_values):+.2f})', fontsize=10)
axes[0].set_xlabel('Opinion')
axes[0].set_ylabel('% of agents')
axes[0].set_xticks(scale_values)

# Each model
for idx, model_name in enumerate(models_list):
    mdf = df[(df["model"] == model_name) & df["llm_numeric"].notna()]
    llm_dist = Counter(mdf["llm_numeric"].astype(int))
    n_m = len(mdf)
    llm_pcts = [llm_dist.get(v, 0) / n_m * 100 for v in scale_values]
    llm_mean = mdf["llm_numeric"].mean()
    
    ax = axes[idx + 1]
    ax.bar(scale_values, llm_pcts, color='coral', alpha=0.8)
    # Overlay ground truth as line
    ax.plot(scale_values, gt_pcts, 'o-', color='steelblue', alpha=0.6, linewidth=1.5, markersize=4)
    ax.set_title(f'{model_name}\n(mean={llm_mean:+.2f})', fontsize=10)
    ax.set_xlabel('Opinion')
    ax.set_xticks(scale_values)

plt.suptitle(f'Day 0 Response Distribution — {POLICY_ID}', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Bias by real survey response (per model)

Shows where each model is worst: for agents whose real answer is −3, what does the model predict?

In [ ]:
# ── Bias breakdown by real response value ──
for model_name in models_list:
    mdf = df[(df["model"] == model_name) & df["llm_numeric"].notna()].copy()
    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'Real':>5} {'n':>3} {'Mean LLM':>9} {'Mean Err':>9} {'Exact%':>7}")
    
    for rv in sorted(mdf["real_numeric"].unique()):
        subset = mdf[mdf["real_numeric"] == rv]
        n_rv = len(subset)
        mean_llm = subset["llm_numeric"].mean()
        mean_err = subset["error"].mean()
        exact_pct = subset["exact_match"].mean() * 100
        print(f"  {rv:+d}  {n_rv:>3}  {mean_llm:>+8.2f}  {mean_err:>+8.2f}  {exact_pct:>6.1f}%")

## Per-agent scatter: LLM vs Real

In [ ]:
# ── Scatter plot: LLM vs Real for each model ──
n_models = len(models_list)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4), sharey=True, sharex=True)
if n_models == 1:
    axes = [axes]

for idx, model_name in enumerate(models_list):
    ax = axes[idx]
    mdf = df[(df["model"] == model_name) & df["llm_numeric"].notna()]
    
    # Add jitter for visibility
    jitter = np.random.default_rng(42).uniform(-0.2, 0.2, size=len(mdf))
    ax.scatter(mdf["real_numeric"] + jitter, mdf["llm_numeric"] + jitter,
               alpha=0.6, s=40, edgecolors='black', linewidths=0.5)
    
    # Perfect prediction line
    ax.plot([-3, 3], [-3, 3], 'k--', alpha=0.3, label='Perfect')
    
    # Compute Spearman for title
    rho_val = summary[summary["Model"] == model_name]["Spearman ρ"].values[0]
    ax.set_title(f"{model_name}\nρ = {rho_val:.3f}", fontsize=10)
    ax.set_xlabel("Real Survey")
    if idx == 0:
        ax.set_ylabel("LLM Day 0")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_xticks(range(-3, 4))
    ax.set_yticks(range(-3, 4))
    ax.set_aspect('equal')

plt.suptitle('LLM Day 0 vs Real Survey Response', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Save results

In [ ]:
# ── Save raw results and summary ──
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"../data/output/experiments/{timestamp}_model_comparison"
os.makedirs(out_dir, exist_ok=True)

df.to_csv(f"{out_dir}/per_agent_results.csv", index=False)
summary.to_csv(f"{out_dir}/model_summary.csv", index=False)

print(f"Results saved to {out_dir}/")
print()
print(summary[display_cols].to_string(index=False))

## Notes

**Metrics guide:**
- **Exact %** — strict: LLM picks the same point on 7-point scale
- **Ordinal score** — lenient: penalises distance proportionally (1 point off = 0.833, 2 off = 0.667, etc.)
- **Mean signed error** — bias indicator: positive = LLM is systematically pro-climate
- **MAE** — average magnitude of error regardless of direction
- **Direction %** — coarse: does the LLM at least get the right side (oppose/neutral/support)?
- **Spearman ρ** — ranking: do agents who *should* be more supportive get higher LLM scores?

**What to look for:**
1. Low mean signed error (near 0) → model has minimal systematic bias
2. High Spearman ρ → model preserves the ordering even if absolute values are off
3. High direction accuracy → model at least gets oppose vs support right
4. Spread across the full −3 to +3 range → model doesn't collapse to a narrow band